CELL 1 ▸ Install libraries

In [ ]:
import subprocess
subprocess.run(["apt-get", "install", "-y", "tesseract-ocr"], capture_output=True)

!pip install -q chromadb sentence-transformers groq flask flask-cors pyngrok PyMuPDF Pillow pytesseract


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

CELL 2 ▸ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
folder = "/content/drive/MyDrive/raw_pdfs"
print("PDFs found:", os.listdir(folder))

Mounted at /content/drive
PDFs found: ['News Papers Clippings Summaries from Business Line.pdf', 'News papers clipping Summaries from Business line, Business standard and Economic Times (20-2-2026).pdf', 'News papers clipping Summaries from Business line, Business standard and Economic Times (21-2-2026).pdf', 'Newspapers Clipping Summaries from Business Line,Business Standard and The Economic Times 23-02-2026.pdf', 'Newspapers Clipping Summaries from Business line, Business Standard and The Economic Times ( 24-2-2026).pdf', 'News paper clippings.pdf', 'Newspapers Summeries from Business line, Business Standard and The Economic Times 26-02-2026.pdf', 'All English Editorial 27-02.pdf', 'Newspaper Clipping summaries 28-02-2025.pdf', 'Newspaper Clipping summaries 27-02-2026.pdf']


CELL 3 ▸ Config with groq

In [ ]:
import os, re, time, json, threading
import fitz
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
PDF_FOLDER    = "/content/drive/MyDrive/raw_pdfs"
CHROMA_DIR    = "/content/chroma_db"
COLLECTION    = "newspaper"
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 100
TOP_K         = 5
MODEL = "llama-3.3-70b-versatile"

print("Config ready ✓")

Config ready ✓


 CELL 4 ▸ Helper functions

In [ ]:
import fitz, re, os
from PIL import Image
import pytesseract

def extract_text(path):
    """Extract text from PDF — handles both text-based and scanned image PDFs"""
    doc = fitz.open(path)
    text = ""
    for i, page in enumerate(doc):
        page_text = page.get_text()
        # If no text found, it's a scanned image — use OCR
        if not page_text.strip():
            pix      = page.get_pixmap(dpi=300)
            img_path = f"/tmp/page_{i}.png"
            pix.save(img_path)
            page_text = pytesseract.image_to_string(
                Image.open(img_path), lang='eng'
            )
            os.remove(img_path)
        text += f"\n[Page {i+1}]\n{page_text}"
    doc.close()
    return re.sub(r'\n{3,}', '\n\n', re.sub(r' {2,}', ' ', text)).strip()

def extract_text_from_image(path):
    """Extract text from image files using OCR"""
    try:
        img = Image.open(path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        text = pytesseract.image_to_string(img, lang='eng')
        if not text.strip():
            return "No readable text found in this image."
        return text.strip()
    except Exception as e:
        return f"Could not extract text from image: {str(e)}"

def chunk_text(text):
    chunks, start = [], 0
    while start < len(text):
        c = text[start:start+CHUNK_SIZE]
        if c.strip():
            chunks.append(c.strip())
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

def load_all_files(folder):
    """Load both PDFs and images from folder"""
    all_docs, all_ids, all_meta = [], [], []

    pdf_files   = [f for f in os.listdir(folder) if f.lower().endswith('.pdf')]
    image_files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]

    print(f"Found {len(pdf_files)} PDFs and {len(image_files)} images")

    for f in pdf_files:
        chunks = chunk_text(extract_text(os.path.join(folder, f)))
        for i, c in enumerate(chunks):
            all_docs.append(c)
            all_ids.append(f"{f}__chunk_{i}")
            all_meta.append({"source": f, "chunk_index": i, "type": "pdf"})
        print(f"  PDF: {f} → {len(chunks)} chunks")

    for f in image_files:
        chunks = chunk_text(extract_text_from_image(os.path.join(folder, f)))
        for i, c in enumerate(chunks):
            all_docs.append(c)
            all_ids.append(f"{f}__chunk_{i}")
            all_meta.append({"source": f, "chunk_index": i, "type": "image"})
        print(f"  Image: {f} → {len(chunks)} chunks")

    print(f"Total: {len(all_docs)} chunks")
    return all_docs, all_ids, all_meta

print("Helpers ready ✓")

Helpers ready ✓


Build vector database

In [ ]:
print("Loading embedding model…")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedder ready ✓")

client = chromadb.PersistentClient(path=CHROMA_DIR)
try: client.delete_collection(COLLECTION)
except: pass
col = client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})


docs, ids, meta = load_all_files(PDF_FOLDER)

print("\nGenerating embeddings and storing…")
BATCH = 50
for i in range(0, len(docs), BATCH):
    emb = embedder.encode(docs[i:i+BATCH]).tolist()
    col.add(documents=docs[i:i+BATCH], embeddings=emb,
            ids=ids[i:i+BATCH], metadatas=meta[i:i+BATCH])
    print(f"  {min(i+BATCH, len(docs))}/{len(docs)} stored")

print(f"\nDatabase ready ✓ ({col.count()} total chunks)")

Loading embedding model…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder ready ✓
Found 10 PDFs and 0 images
  PDF: News Papers Clippings Summaries from Business Line.pdf → 32 chunks
  PDF: News papers clipping Summaries from Business line, Business standard and Economic Times (20-2-2026).pdf → 11 chunks
  PDF: News papers clipping Summaries from Business line, Business standard and Economic Times (21-2-2026).pdf → 8 chunks
  PDF: Newspapers Clipping Summaries from Business Line,Business Standard and The Economic Times 23-02-2026.pdf → 10 chunks
  PDF: Newspapers Clipping Summaries from Business line, Business Standard and The Economic Times ( 24-2-2026).pdf → 11 chunks
  PDF: News paper clippings.pdf → 10 chunks
  PDF: Newspapers Summeries from Business line, Business Standard and The Economic Times 26-02-2026.pdf → 9 chunks
  PDF: All English Editorial 27-02.pdf → 848 chunks
  PDF: Newspaper Clipping summaries 28-02-2025.pdf → 7 chunks
  PDF: Newspaper Clipping summaries 27-02-2026.pdf → 8 chunks
Total: 954 chunks

Generating embeddings and storin

CELL 6 ▸ RAG engine

In [ ]:
from groq import Groq
import re

groq_client = Groq(api_key=GROQ_API_KEY)


def correct_query(query):
    """Auto-correct spelling and grammar in user query before searching"""
    r = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content":
             "You are a spelling and grammar corrector. "
             "Correct any spelling mistakes or grammar errors in the user's query. "
             "Return ONLY the corrected query, nothing else. "
             "Do not change the meaning. If query is already correct, return it as is."},
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=100
    )
    return r.choices[0].message.content.strip()


def retrieve(query):
    corrected = correct_query(query)
    print(f"Original : {query}")
    print(f"Corrected: {corrected}")
    emb = embedder.encode(corrected).tolist()
    res = col.query(
        query_embeddings=emb,
        n_results=TOP_K,
        include=["documents", "metadatas"]
    )
    parts, sources = [], set()
    for chunk, m in zip(res["documents"][0], res["metadatas"][0]):
        parts.append(f"[From: {m['source']}]\n{chunk}")
        sources.add(m["source"])
    return "\n\n--\n\n".join(parts), list(sources), corrected


def ask(question, context, history=None, corrected_query=None):
    sys_prompt = (
        "You are a Smart Newspaper Assistant with memory of the entire conversation.\n"
        "Answer ONLY from the provided newspaper context.\n"
        "If the answer is not in the context say: "
        "'I couldn't find that in the uploaded newspapers.'\n"
        "IMPORTANT: You remember everything discussed in this conversation.\n"
        "When the user asks follow-up questions like 'tell me more', 'explain that', "
        "'what about...' — always refer back to previous messages to understand what they mean.\n"
        "Be clear, friendly, detailed and concise. Mention the source edition when relevant."
    )
    messages = [{"role": "system", "content": sys_prompt}]


    if history:
        messages += history


    user_content = f"Context:\n{context}\n\nQuestion: {question}"
    if corrected_query and corrected_query.lower() != question.lower():
        user_content = (
            f"Context:\n{context}\n\n"
            f"[Note: User wrote '{question}', interpreted as '{corrected_query}']\n"
            f"Question: {corrected_query}"
        )

    messages.append({"role": "user", "content": user_content})

    r = groq_client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=0.3, max_tokens=1024
    )
    return r.choices[0].message.content

def summarize(pdf_name):
    r = col.get(where={"source": pdf_name}, include=["documents"])
    if not r["documents"]:
        return f"No data found for {pdf_name}."
    text = "\n".join(r["documents"][0])
    r2 = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "Summarize this newspaper edition. "
                        "Use bullet points for key topics, events, and highlights."},
            {"role": "user", "content": text}
        ],
        temperature=0.3, max_tokens=800
    )
    return r2.choices[0].message.content

def add_file(path):
    name = os.path.basename(path)
    ext  = name.lower().split('.')[-1]

    if ext == 'pdf':
        text  = extract_text(path)
        ftype = 'pdf'
    elif ext in ['jpg', 'jpeg', 'png', 'webp']:
        text  = extract_text_from_image(path)
        ftype = 'image'
    else:
        return 0

    chunks = chunk_text(text)
    ids_   = [f"{name}__chunk_{i}" for i in range(len(chunks))]
    metas_ = [{"source": name, "chunk_index": i, "type": ftype}
              for i in range(len(chunks))]
    emb    = embedder.encode(chunks).tolist()
    col.add(documents=chunks, embeddings=emb, ids=ids_, metadatas=metas_)
    return len(chunks)

print("RAG engine ready ✓")

RAG engine ready ✓


Quick test

In [ ]:
ctx, srcs, corrected = retrieve("What is in the newspaper?")
ans = ask("What is in the newspaper?", ctx)
print("Answer:", ans)
print("Sources:", srcs)
print("Corrected query:", corrected)

Original : What is in the newspaper?
Corrected: What is in the newspaper?
Answer: The newspaper, All English Editorial 27-02.pdf, contains various articles and news items, including:

1. A discussion on the impact of AI on journalism, particularly the use of Large Language Models (LLMs) and their potential to disrupt the profession.
2. A report on the visit of women farmers from Dausa district, Rajasthan, to the Pusa Krishi Vigyan Mela at the Indian Agricultural Research Institute grounds in New Delhi.
3. A mention of Pakistan's response to Canada, although the context is not fully provided.
4. An article on the clampdown on newspapers in Hong Kong, including the prosecution of Jimmy Lai, the founder of Apple Daily, under the National Security Law (NSL).
5. A commentary on the importance of formal language and truth in official texts, such as school textbooks, and the need for a minimalist approach in these materials.

These are just a few examples of the content found in the newspaper

Flask backend  (serves both API + the UI page)

In [ ]:
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from datetime import datetime
import json

app      = Flask(__name__)
CORS(app)
sessions   = {}
UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ✅ Persistent history setup
HISTORY_FILE = "/content/drive/MyDrive/chat_history.json"

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_history(history):
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

persistent_history = load_history()
print(f"Loaded {len(persistent_history)} history items ✓")

@app.route("/")
def index():
    return Response(HTML_PAGE, mimetype="text/html")

@app.route("/api/health")
def health():
    return jsonify({"status":"ok","docs_in_db": col.count()})

@app.route("/api/editions")
def editions():
    all_meta = col.get(include=["metadatas"])
    srcs = sorted({m["source"] for m in all_meta["metadatas"]})
    return jsonify({"editions": srcs, "total_chunks": col.count()})

# ✅ UPDATED: saves every Q&A to persistent history file
@app.route("/api/query", methods=["POST"])
def query():
    global persistent_history
    d   = request.json
    q   = d.get("question","").strip()
    sid = d.get("session_id","default")
    if not q:
        return jsonify({"error":"empty"}), 400

    sessions.setdefault(sid, [])
    ctx, srcs, corrected = retrieve(q)
    ans = ask(q, ctx, sessions[sid], corrected)

    sessions[sid] += [
        {"role": "user",      "content": corrected},
        {"role": "assistant", "content": ans}
    ]

    # ✅ Save to persistent history
    persistent_history.append({
        "q":         q,
        "corrected": corrected,
        "answer":    ans,
        "sources":   srcs,
        "time":      datetime.now().strftime("%d %b %Y, %I:%M %p")
    })
    save_history(persistent_history)

    return jsonify({
        "answer":    ans,
        "sources":   srcs,
        "corrected": corrected
    })

@app.route("/api/summarize", methods=["POST"])
def do_summarize():
    name = request.json.get("pdf_name","")
    if not name:
        return jsonify({"error":"no pdf_name"}), 400
    return jsonify({"summary": summarize(name), "pdf_name": name})

@app.route("/api/upload", methods=["POST"])
def upload():
    f = request.files.get("file")
    if not f:
        return jsonify({"error": "no file"}), 400
    allowed = ('.pdf', '.jpg', '.jpeg', '.png', '.webp')
    if not f.filename.lower().endswith(allowed):
        return jsonify({"error": "PDF or image only (jpg, png, webp)"}), 400
    path = os.path.join(UPLOAD_DIR, f.filename)
    f.save(path)
    n = add_file(path)
    return jsonify({"message": f"Added {f.filename}",
                    "chunks_added": n,
                    "total_in_db": col.count()})

@app.route("/api/clear_session", methods=["POST"])
def clear():
    sid = request.json.get("session_id","default")
    sessions[sid] = []
    return jsonify({"ok": True})

# ✅ NEW: get full persistent history
@app.route("/api/history", methods=["GET"])
def get_history():
    return jsonify({"history": persistent_history})

# ✅ NEW: clear persistent history
@app.route("/api/history/clear", methods=["POST"])
def clear_persistent_history():
    global persistent_history
    persistent_history = []
    save_history(persistent_history)
    return jsonify({"ok": True})

print("Flask app defined ✓")

Loaded 22 history items ✓
Flask app defined ✓


CELL 9 ▸ HTML_PAGE

In [ ]:
HTML_PAGE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>Smart Newspaper Assistant</title>
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet"/>
<style>
:root{
  --bg:#0c0f1a;--bg2:#13172a;--bg3:#1c2138;--surface:#222846;
  --border:#2e3556;--accent:#6c7cf7;--accent2:#a78bfa;--gold:#f5c842;
  --text:#e8eaf6;--muted:#8891b8;--danger:#f87171;--success:#4ade80;
  --fh:'Playfair Display',serif;--fb:'DM Sans',sans-serif;
}
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{font-family:var(--fb);background:var(--bg);color:var(--text);
  min-height:100vh;display:flex;flex-direction:column;overflow-x:hidden}
body::before{content:'';position:fixed;inset:0;pointer-events:none;z-index:0;
  background:radial-gradient(ellipse 70% 50% at 10% 20%,rgba(108,124,247,.08) 0%,transparent 60%),
             radial-gradient(ellipse 50% 40% at 90% 80%,rgba(167,139,250,.07) 0%,transparent 60%)}
/* HEADER */
header{position:relative;z-index:10;background:rgba(19,23,42,.97);
  backdrop-filter:blur(12px);border-bottom:1px solid var(--border);
  padding:0 20px;height:60px;display:flex;align-items:center;justify-content:space-between;gap:10px}
.logo{width:36px;height:36px;background:linear-gradient(135deg,var(--accent),var(--accent2));
  border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:17px;flex-shrink:0}
.htitle h1{font-family:var(--fh);font-size:15px;font-weight:700;line-height:1.2}
.htitle p{font-size:10px;color:var(--muted)}
.hactions{display:flex;gap:8px;align-items:center}
/* LAYOUT */
.layout{position:relative;z-index:1;display:grid;grid-template-columns:250px 1fr;
  flex:1;height:calc(100vh - 60px);overflow:hidden}
/* SIDEBAR */
aside{background:var(--bg2);border-right:1px solid var(--border);
  display:flex;flex-direction:column;overflow:hidden}
.ss{padding:14px;border-bottom:1px solid var(--border)}
.ss h3{font-size:9px;font-weight:600;text-transform:uppercase;letter-spacing:1.2px;
  color:var(--muted);margin-bottom:8px}
.elist{list-style:none;max-height:190px;overflow-y:auto;scrollbar-width:thin;scrollbar-color:var(--border) transparent}
.elist li{font-size:11px;color:var(--muted);padding:5px 8px;border-radius:6px;
  cursor:pointer;transition:all .15s;display:flex;align-items:center;gap:5px}
.elist li:hover{background:var(--surface);color:var(--text)}
.elist li::before{content:'📰';font-size:10px}
.btn-sum{width:100%;margin-top:8px;padding:7px;background:rgba(108,124,247,.12);
  border:1px solid rgba(108,124,247,.3);border-radius:8px;color:var(--accent2);
  font-size:11px;font-weight:500;cursor:pointer;transition:all .2s;font-family:var(--fb)}
.btn-sum:hover{background:rgba(108,124,247,.22)}
/* Upload */
.uzone{border:1.5px dashed var(--border);border-radius:10px;padding:12px;
  text-align:center;cursor:pointer;transition:all .2s}
.uzone:hover{border-color:var(--accent);background:rgba(108,124,247,.05)}
.uzone p{font-size:11px;color:var(--muted);line-height:1.5}
.uzone strong{color:var(--accent2)}
#upload-input{display:none}
#upload-status{font-size:11px;margin-top:5px;min-height:14px;text-align:center}
/* Stats */
.stat{display:flex;justify-content:space-between;font-size:11px;padding:3px 0;color:var(--muted)}
.stat span:last-child{color:var(--accent2);font-weight:600}
/* CHAT */
main{display:flex;flex-direction:column;overflow:hidden}
.cwin{flex:1;overflow-y:auto;padding:22px;display:flex;flex-direction:column;gap:18px;
  scrollbar-width:thin;scrollbar-color:var(--border) transparent}
.welcome{display:flex;flex-direction:column;align-items:center;justify-content:center;
  flex:1;text-align:center;padding:40px;gap:10px}
.welcome .icon{font-size:52px;line-height:1;margin-bottom:6px}
.welcome h2{font-family:var(--fh);font-size:26px;font-weight:700}
.welcome p{font-size:13px;color:var(--muted);max-width:380px;line-height:1.6}
.wpills{display:flex;gap:8px;flex-wrap:wrap;justify-content:center;margin-top:6px}
.wpill{background:var(--bg3);border:1px solid var(--border);border-radius:20px;
  padding:4px 11px;font-size:11px;color:var(--muted)}
/* Messages */
.msg{display:flex;gap:10px;animation:fs .3s ease;max-width:780px}
@keyframes fs{from{opacity:0;transform:translateY(7px)}to{opacity:1;transform:translateY(0)}}
.msg.user{flex-direction:row-reverse;align-self:flex-end}
.mavatar{width:32px;height:32px;border-radius:50%;display:flex;align-items:center;
  justify-content:center;font-size:14px;flex-shrink:0;margin-top:2px}
.msg.user .mavatar{background:linear-gradient(135deg,var(--accent),var(--accent2))}
.msg.bot  .mavatar{background:linear-gradient(135deg,#2e3566,#3d4a80);border:1px solid var(--border)}
.mbody{flex:1}
.mbubble{display:inline-block;padding:11px 15px;border-radius:15px;font-size:13.5px;
  line-height:1.65;max-width:100%}
.msg.user .mbubble{background:linear-gradient(135deg,rgba(108,124,247,.25),rgba(167,139,250,.18));
  border:1px solid rgba(108,124,247,.3);border-top-right-radius:4px}
.msg.bot  .mbubble{background:var(--bg3);border:1px solid var(--border);border-top-left-radius:4px}
.msrcs{margin-top:5px;display:flex;flex-wrap:wrap;gap:4px}
.stag{background:rgba(245,200,66,.1);border:1px solid rgba(245,200,66,.25);
  color:var(--gold);font-size:10px;padding:2px 7px;border-radius:20px;font-weight:500}
.typing{display:flex;align-items:center;gap:4px;padding:11px 15px;
  background:var(--bg3);border:1px solid var(--border);
  border-radius:15px;border-top-left-radius:4px;width:fit-content}
.tdot{width:6px;height:6px;background:var(--muted);border-radius:50%;animation:b 1.2s infinite}
.tdot:nth-child(2){animation-delay:.2s}.tdot:nth-child(3){animation-delay:.4s}
@keyframes b{0%,80%,100%{transform:translateY(0);opacity:.5}40%{transform:translateY(-5px);opacity:1}}
/* Modal */
.overlay{position:fixed;inset:0;background:rgba(0,0,0,.72);z-index:100;
  display:flex;align-items:center;justify-content:center;padding:20px;
  backdrop-filter:blur(4px);opacity:0;pointer-events:none;transition:opacity .2s}
.overlay.open{opacity:1;pointer-events:all}
.modal{background:var(--bg2);border:1px solid var(--border);border-radius:18px;
  width:100%;max-width:580px;max-height:82vh;display:flex;flex-direction:column;
  overflow:hidden;transform:translateY(18px);transition:transform .2s}
.overlay.open .modal{transform:translateY(0)}
.mhead{padding:16px 20px;border-bottom:1px solid var(--border);
  display:flex;align-items:center;justify-content:space-between}
.mhead h2{font-family:var(--fh);font-size:17px;font-weight:700}
.msel{padding:11px 20px;border-bottom:1px solid var(--border);display:flex;gap:8px;align-items:center}
.msel select{flex:1;background:var(--bg3);border:1px solid var(--border);color:var(--text);
  padding:6px 10px;border-radius:8px;font-size:12px;font-family:var(--fb);outline:none}
.mbody2{padding:20px;overflow-y:auto;font-size:13.5px;line-height:1.7;
  color:var(--text);white-space:pre-wrap}
/* Input bar */
.ibar{padding:14px 18px;background:var(--bg2);border-top:1px solid var(--border);
  display:flex;align-items:flex-end;gap:9px}
.iwrap{flex:1;background:var(--bg3);border:1.5px solid var(--border);border-radius:13px;
  padding:9px 13px;display:flex;align-items:flex-end;gap:7px;transition:border-color .2s}
.iwrap:focus-within{border-color:var(--accent)}
#ui{flex:1;background:none;border:none;outline:none;color:var(--text);
  font-family:var(--fb);font-size:13.5px;resize:none;max-height:110px;line-height:1.5;padding:0}
#ui::placeholder{color:var(--muted)}
.ibtn{background:none;border:none;cursor:pointer;padding:4px;border-radius:6px;
  color:var(--muted);font-size:16px;display:flex;align-items:center;justify-content:center;
  transition:color .15s,background .15s;flex-shrink:0}
.ibtn:hover{color:var(--text);background:var(--surface)}
.sbtn{width:42px;height:42px;background:linear-gradient(135deg,var(--accent),var(--accent2));
  border:none;border-radius:12px;cursor:pointer;display:flex;align-items:center;
  justify-content:center;font-size:17px;transition:opacity .15s,transform .1s;flex-shrink:0}
.sbtn:hover{opacity:.9;transform:scale(1.04)}.sbtn:disabled{opacity:.4;cursor:not-allowed;transform:none}
/* Util */
::-webkit-scrollbar{width:4px}::-webkit-scrollbar-track{background:transparent}
::-webkit-scrollbar-thumb{background:var(--border);border-radius:99px}
.btn{padding:7px 14px;border-radius:8px;font-size:11px;font-weight:500;cursor:pointer;
  border:none;font-family:var(--fb);transition:all .15s}
.btn-p{background:var(--accent);color:#fff}.btn-p:hover{background:#5a6ae0}
.btn-g{background:transparent;border:1px solid var(--border);color:var(--muted)}
.btn-g:hover{border-color:var(--accent);color:var(--text)}
.spin{width:14px;height:14px;border:2px solid var(--border);border-top-color:var(--accent);
  border-radius:50%;animation:sp .7s linear infinite;display:inline-block}
@keyframes sp{to{transform:rotate(360deg)}}
#vbtn.active{color:var(--danger)!important;animation:pulse 1s infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.5}}
@media(max-width:660px){.layout{grid-template-columns:1fr}aside{display:none}}
/* HISTORY PANEL */
.hist-panel{position:fixed;right:0;top:60px;height:calc(100vh - 60px);
  width:280px;background:var(--bg2);border-left:1px solid var(--border);
  display:flex;flex-direction:column;transform:translateX(100%);
  transition:transform .3s ease;z-index:50}
.hist-panel.open{transform:translateX(0)}
.hist-header{padding:14px;border-bottom:1px solid var(--border);
  display:flex;align-items:center;justify-content:space-between}
.hist-header h3{font-size:11px;font-weight:600;text-transform:uppercase;
  letter-spacing:1.2px;color:var(--muted)}
.hist-list{flex:1;overflow-y:auto;padding:10px;display:flex;
  flex-direction:column;gap:6px;scrollbar-width:thin;
  scrollbar-color:var(--border) transparent}
.hist-item{background:var(--bg3);border:1px solid var(--border);
  border-radius:8px;padding:8px 10px;cursor:pointer;transition:all .15s}
.hist-item:hover{border-color:var(--accent);background:var(--surface)}
.hist-q{font-size:11px;color:var(--text);line-height:1.4;
  white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.hist-a{font-size:10px;color:var(--muted);margin-top:3px;
  white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.hist-time{font-size:9px;color:var(--muted);margin-top:4px}
.hist-empty{font-size:11px;color:var(--muted);text-align:center;
  padding:20px;line-height:1.6}
.hist-clear{width:calc(100% - 28px);margin:10px 14px;padding:7px;
  background:rgba(248,113,113,.1);border:1px solid rgba(248,113,113,.3);
  border-radius:8px;color:var(--danger);font-size:11px;font-weight:500;
  cursor:pointer;font-family:var(--fb);transition:all .2s}
.hist-clear:hover{background:rgba(248,113,113,.2)}
#hist-toggle{position:fixed;right:0;top:50%;transform:translateY(-50%);
  background:var(--accent);border:none;border-radius:8px 0 0 8px;
  padding:10px 6px;cursor:pointer;z-index:51;
  writing-mode:vertical-rl;font-size:10px;font-weight:600;
  color:#fff;font-family:var(--fb);letter-spacing:1px;
  transition:background .15s}
#hist-toggle:hover{background:#5a6ae0}
</style>
</head>
<body>

<header>
  <div style="display:flex;align-items:center;gap:10px">
    <div class="logo">📰</div>
    <div class="htitle">
      <h1>Smart Newspaper Assistant</h1>
      <p>RAG-powered search over your uploaded editions</p>
    </div>
  </div>
  <div class="hactions">
    <button class="btn btn-g" onclick="clearSession()">🗑 Clear</button>
    <button class="btn btn-p" onclick="openModal()">📋 Summarize</button>
  </div>
</header>

<div class="layout">
  <!-- SIDEBAR -->
  <aside>
    <div class="ss">
      <h3>Database</h3>
      <div class="stat"><span>Editions</span><span id="st-ed">—</span></div>
      <div class="stat"><span>Chunks</span><span id="st-ch">—</span></div>
      <div class="stat"><span>Status</span><span id="st-s">Loading…</span></div>
    </div>
    <div class="ss" style="flex:0 0 auto">
      <h3>Newspaper Editions</h3>
      <ul class="elist" id="elist"><li style="cursor:default;font-size:11px">Loading…</li></ul>
      <button class="btn-sum" onclick="openModal()">📋 Summarize an edition</button>
    </div>
    <div class="ss" style="flex:1">
      <h3>Add New PDF or Image</h3>
      <div class="uzone" onclick="document.getElementById('upload-input').click()">
        <p>Click to upload a newspaper<br/>
        <strong>PDF or Image</strong><br/>
        <span style="font-size:10px;color:var(--muted)">jpg · png · webp · pdf</span></p>
      </div>
      <input type="file" id="upload-input"
        accept=".pdf,.jpg,.jpeg,.png,.webp"
        onchange="uploadPDF(this)"/>
      <div id="upload-status"></div>
    </div>
  </aside>

  <!-- CHAT -->
  <main>
    <div class="cwin" id="cwin">
      <div class="welcome" id="ws">
        <div class="icon">📰</div>
        <h2>Smart Newspaper Assistant</h2>
        <p>Ask anything from your uploaded newspaper editions. Answers come only from your indexed PDF data.</p>
        <div class="wpills">
          <span class="wpill">🔍 Semantic search</span>
          <span class="wpill">🎙 Voice input</span>
          <span class="wpill">📋 Summarize editions</span>
          <span class="wpill">📤 Upload PDF & Images</span>
        </div>
      </div>
    </div>
    <div class="ibar">
      <div class="iwrap">
        <textarea id="ui" rows="1" placeholder="Ask anything from your uploaded newspapers…"
          onkeydown="hk(event)" oninput="ar(this)"></textarea>
        <button class="ibtn" id="vbtn" onclick="tV()" title="Voice input (Chrome only)">🎙</button>
      </div>
      <button class="sbtn" id="sbtn" onclick="send()">
        <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round">
          <line x1="22" y1="2" x2="11" y2="13"/><polygon points="22 2 15 22 11 13 2 9 22 2"/>
        </svg>
      </button>
    </div>
  </main>
</div>

<!-- SUMMARY MODAL -->
<div class="overlay" id="modal">
  <div class="modal">
    <div class="mhead">
      <h2>📋 Edition Summary</h2>
      <button class="btn btn-g" onclick="closeModal()">✕ Close</button>
    </div>
    <div class="msel">
      <select id="esel"><option value="">— Select edition —</option></select>
      <button class="btn btn-p" onclick="doSummary()">Summarize</button>
    </div>
    <div class="mbody2" id="mbody">Select an edition above and click Summarize.</div>
  </div>
</div>

<!-- HISTORY TOGGLE BUTTON -->
<button id="hist-toggle" onclick="toggleHist()">HISTORY</button>

<!-- HISTORY PANEL -->
<div class="hist-panel" id="hist-panel">
  <div class="hist-header">
    <h3>💬 Chat History</h3>
    <button class="btn btn-g" onclick="toggleHist()">✕</button>
  </div>
  <div class="hist-list" id="hist-list">
    <div class="hist-empty">No history yet.<br/>Start asking questions!</div>
  </div>
  <button class="hist-clear" onclick="clearHistory()">🗑 Clear History</button>
</div>

<script>
const BASE = '';
let SID = 'sid_' + Math.random().toString(36).slice(2);
let busy = false, recog = null, isRec = false;
let chatHistory = [];

// ── boot ✅ UPDATED: loads persistent history on startup ──────
(async()=>{
  try{
    const r = await fetch(BASE+'/api/health');
    const d = await r.json();
    document.getElementById('st-ch').textContent = d.docs_in_db;
    document.getElementById('st-s').textContent  = 'Online ✓';
    document.getElementById('st-s').style.color  = 'var(--success)';
    await loadEditions();
    await loadPersistentHistory(); // ✅ load saved history from server
  }catch{
    document.getElementById('st-s').textContent = 'Error';
    document.getElementById('st-s').style.color = 'var(--danger)';
  }
})();

// ✅ NEW: load history from server file
async function loadPersistentHistory(){
  try{
    const r = await fetch(BASE+'/api/history');
    const d = await r.json();
    // reverse so newest is first
    chatHistory = [...d.history].reverse();
    renderHistory();
  }catch(e){
    console.log('History load failed:', e);
  }
}

async function loadEditions(){
  const r = await fetch(BASE+'/api/editions');
  const d = await r.json();
  document.getElementById('st-ed').textContent = d.editions.length;
  document.getElementById('st-ch').textContent = d.total_chunks;
  const el = document.getElementById('elist');
  const ms = document.getElementById('esel');
  el.innerHTML = '';
  ms.innerHTML = '<option value="">— Select edition —</option>';
  d.editions.forEach(ed=>{
    const li=document.createElement('li');
    li.textContent=ed; li.onclick=()=>sq(`Summarize the news in ${ed}`);
    el.appendChild(li);
    const op=document.createElement('option');
    op.value=op.textContent=ed; ms.appendChild(op);
  });
}

// ── send ──────────────────────────────────────────────────────
async function send(){
  const inp=document.getElementById('ui');
  const q=inp.value.trim();
  if(!q||busy)return;
  inp.value=''; ar(inp);
  rmWelcome(); addMsg('user',q);
  const tid=addTyping();
  busy=true; document.getElementById('sbtn').disabled=true;
  try{
    const r=await fetch(BASE+'/api/query',{
      method:'POST',headers:{'Content-Type':'application/json'},
      body:JSON.stringify({question:q,session_id:SID})});
    const d=await r.json();
    rmTyping(tid);
    if(d.corrected && d.corrected.toLowerCase() !== q.toLowerCase()){
      addMsg('bot', `✏️ I corrected your query to: "${d.corrected}"`, []);
    }
    addMsg('bot', d.answer, d.sources||[]);
    addToHistory(q, d.corrected || q, d.answer, d.time || '');
  }catch{
    rmTyping(tid);
    addMsg('bot','⚠️ Backend error. Make sure all Colab cells ran without errors.',[]);
  }
  busy=false; document.getElementById('sbtn').disabled=false;
}
function sq(q){document.getElementById('ui').value=q;send();}
function hk(e){if(e.key==='Enter'&&!e.shiftKey){e.preventDefault();send();}}

// ── messages ──────────────────────────────────────────────────
function rmWelcome(){const w=document.getElementById('ws');if(w)w.remove();}
function addMsg(role,text,srcs=[]){
  const cwin=document.getElementById('cwin');
  const d=document.createElement('div');
  d.className='msg '+role;
  const av=role==='user'?'👤':'🤖';
  const sh=srcs.length?'<div class="msrcs">'+srcs.map(s=>`<span class="stag">📰 ${s}</span>`).join('')+'</div>':'';
  d.innerHTML=`<div class="mavatar">${av}</div>
    <div class="mbody"><div class="mbubble">${esc(text).replace(/\n/g,'<br/>')}</div>${sh}</div>`;
  cwin.appendChild(d); cwin.scrollTop=cwin.scrollHeight;
}
let tc=0;
function addTyping(){
  const id='t'+(++tc), cwin=document.getElementById('cwin');
  const d=document.createElement('div');
  d.className='msg bot'; d.id=id;
  d.innerHTML=`<div class="mavatar">🤖</div>
    <div class="mbody"><div class="typing"><div class="tdot"></div><div class="tdot"></div><div class="tdot"></div></div></div>`;
  cwin.appendChild(d); cwin.scrollTop=cwin.scrollHeight; return id;
}
function rmTyping(id){const e=document.getElementById(id);if(e)e.remove();}
function esc(t){return t.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');}

// ── clear ✅ UPDATED: clears both session and persistent history
async function clearSession(){
  try{await fetch(BASE+'/api/clear_session',{method:'POST',
    headers:{'Content-Type':'application/json'},body:JSON.stringify({session_id:SID})});}catch{}
  SID='sid_'+Math.random().toString(36).slice(2);
  document.getElementById('cwin').innerHTML=`<div class="welcome" id="ws">
    <div class="icon">📰</div><h2>Smart Newspaper Assistant</h2>
    <p>Ask anything from your uploaded newspaper editions.</p>
    <div class="wpills"><span class="wpill">🔍 Semantic search</span>
    <span class="wpill">🎙 Voice input</span><span class="wpill">📋 Summarize</span></div></div>`;
  chatHistory = [];
  renderHistory();
}

// ── voice ─────────────────────────────────────────────────────
function tV(){
  const btn = document.getElementById('vbtn');
  if(!('webkitSpeechRecognition' in window || 'SpeechRecognition' in window)){
    btn.textContent = '❌';
    setTimeout(()=>btn.textContent='🎙', 2000);
    addMsg('bot','⚠️ Voice only works in Chrome browser. Please open this URL in Chrome.',[]);
    return;
  }
  navigator.mediaDevices.getUserMedia({audio:true})
  .then(()=>{
    if(isRec){ recog.stop(); return; }
    const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
    recog = new SR();
    recog.lang = 'en-IN';
    recog.continuous = false;
    recog.interimResults = false;
    recog.onstart = ()=>{
      isRec = true;
      btn.classList.add('active');
      btn.textContent = '⏹';
      addMsg('bot','🎙 Listening... speak now!', []);
    };
    recog.onresult = e=>{
      const t = e.results[0][0].transcript;
      document.getElementById('ui').value = t;
      ar(document.getElementById('ui'));
      addMsg('bot', `✓ Heard: "${t}" — sending...`, []);
      setTimeout(()=>send(), 500);
    };
    recog.onerror = e=>{
      addMsg('bot', `⚠️ Voice error: ${e.error}. Try again or type instead.`, []);
      isRec = false;
      btn.classList.remove('active');
      btn.textContent = '🎙';
    };
    recog.onend = ()=>{
      isRec = false;
      btn.classList.remove('active');
      btn.textContent = '🎙';
    };
    recog.start();
  })
  .catch(()=>{
    addMsg('bot','⚠️ Microphone permission denied. Click the 🔒 icon in your browser address bar and allow microphone access, then try again.',[]);
  });
}

// ── upload ────────────────────────────────────────────────────
async function uploadPDF(inp){
  const file=inp.files[0]; if(!file)return;
  const st=document.getElementById('upload-status');
  st.innerHTML='<span class="spin"></span> Uploading…';
  const fd=new FormData(); fd.append('file',file);
  try{
    const r=await fetch(BASE+'/api/upload',{method:'POST',body:fd});
    const d=await r.json();
    st.style.color='var(--success)';
    st.textContent=`✓ Added ${d.chunks_added} chunks`;
    await loadEditions();
  }catch{st.style.color='var(--danger)';st.textContent='✗ Upload failed';}
  inp.value='';
}

// ── modal ─────────────────────────────────────────────────────
function openModal(){document.getElementById('modal').classList.add('open');}
function closeModal(){document.getElementById('modal').classList.remove('open');}
async function doSummary(){
  const nm=document.getElementById('esel').value;
  const mb=document.getElementById('mbody');
  if(!nm){mb.textContent='Please select an edition.';return;}
  mb.innerHTML='<span class="spin"></span> Generating summary…';
  try{
    const r=await fetch(BASE+'/api/summarize',{method:'POST',
      headers:{'Content-Type':'application/json'},body:JSON.stringify({pdf_name:nm})});
    const d=await r.json(); mb.textContent=d.summary;
  }catch{mb.textContent='Failed to load summary.';}
}
document.getElementById('modal').addEventListener('click',function(e){if(e.target===this)closeModal();});

// ── textarea resize ───────────────────────────────────────────
function ar(el){el.style.height='auto';el.style.height=Math.min(el.scrollHeight,110)+'px';}

// ── history panel functions ───────────────────────────────────
function toggleHist(){
  document.getElementById('hist-panel').classList.toggle('open');
}

// ✅ UPDATED: accepts time from server too
function addToHistory(q, corrected, answer, serverTime=''){
  const now  = new Date();
  const time = serverTime || now.toLocaleTimeString('en-IN',{hour:'2-digit',minute:'2-digit'});
  chatHistory.unshift({q, corrected, answer, time});
  renderHistory();
}

function renderHistory(){
  const hl = document.getElementById('hist-list');
  if(chatHistory.length === 0){
    hl.innerHTML = '<div class="hist-empty">No history yet.<br/>Start asking questions!</div>';
    return;
  }
  hl.innerHTML = chatHistory.map((item, idx) => `
    <div class="hist-item" onclick="reaskQuestion(${idx})">
      <div class="hist-q">🙋 ${esc(item.corrected || item.q)}</div>
      <div class="hist-a">🤖 ${esc(item.answer.slice(0,80))}${item.answer.length>80?'...':''}</div>
      <div class="hist-time">🕐 ${item.time}</div>
    </div>
  `).join('');
}

function reaskQuestion(idx){
  const item = chatHistory[idx];
  document.getElementById('ui').value = item.corrected || item.q;
  ar(document.getElementById('ui'));
  toggleHist();
  send();
}

// ✅ UPDATED: clears from server file too
async function clearHistory(){
  chatHistory = [];
  renderHistory();
  try{
    await fetch(BASE+'/api/history/clear', {method:'POST'});
  }catch(e){
    console.log('Clear history failed:', e);
  }
}
</script>
</body>
</html>'''

print("HTML page string defined ✓")

HTML page string defined ✓


CELL 10 ▸ START SERVER + OPEN BROWSER  ← run this last

In [ ]:
import threading, time, os

os.system("kill -9 $(lsof -t -i:5000) 2>/dev/null || true")
time.sleep(1)


os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
os.system("chmod +x cloudflared")

def run():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(2)

# ── Start Cloudflare tunnel ───────────────────────────────────
import subprocess, re

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print("Waiting for tunnel URL...")
for line in tunnel.stdout:
    line = line.decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print("=" * 55)
        print("  SERVER IS LIVE!")
        print(f"  Open this in your browser:")
        print(f"  {url}")
        print("=" * 55)
        break

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Waiting for tunnel URL...
  SERVER IS LIVE!
  Open this in your browser:
  https://distribute-lawrence-underground-trademarks.trycloudflare.com
